# 03 — Statistical Association Between Early Behavioral Signals and Eventual Default

## Purpose

This notebook moves from **exploration to formal statistical investigation**.

Notebook 02 identified candidate behavioral signals through exploratory analysis. Here, the objective is to test whether those candidate variables are statistically associated with the eventual default outcome.

The analysis is intentionally framed as **association, not causation**.

### Questions

1. Is early missed-installment behavior associated with eventual default?
2. Is persistent/consecutive delinquency associated with eventual default?
3. Is delayed repayment recovery associated with eventual default?
4. Does normalized overdue persistence show an association with eventual default?
5. Is relative overdue burden associated with eventual default?
6. Which categorical dimensions show evidence of association with the outcome?

The eventual target remains:

- `0` — eventually settles within original duration
- `1` — eventually defaults

Only early-life behavioral variables are considered as candidate predictors.

## 1. Statistical design

The candidate predictor types determine the appropriate analysis:

| Predictor type | Outcome | Primary exploratory test |
|---|---|---|
| Continuous | Binary default | Point-biserial correlation |
| Categorical | Binary default | Chi-square test of independence |
| Ordered/binned behavior | Binary default | Contingency-table analysis |

### Interpretation rule

A statistically significant association does **not** imply that the variable causes default.

The purpose of this stage is to establish evidence that a behavioral signal is worth carrying into feature engineering and predictive modeling.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pointbiserialr, chi2_contingency

DATA_PATH = Path("../data/synthetic/synthetic_early_modeling_base.csv")
df = pd.read_csv(DATA_PATH)

df.shape

## 2. Reconstruct candidate behavioral variables

The variables below correspond to the behavioral representations investigated during the exploratory stage.

The synthetic dataset contains early-life measures only for modeling purposes. Any full-lifecycle variables used in data generation are excluded from the statistical predictor set.

In [ ]:
frequency_days = {
    "Weekly": 7,
    "Bi-weekly": 14,
    "Monthly": 28
}

df["pass_due_cycle_ratio"] = (
    df["early_max_overdue_days"]
    / df["installment_days"]
)

df["overdue_installment_equivalent"] = (
    df["early_total_overdue_days"]
    / df["installment_days"]
)

df["missed_installment_proportion"] = (
    df["early_missed_installment_count"]
    / df["prediction_installment"].clip(lower=1)
)

df["has_consecutive_miss"] = (
    df["early_max_consecutive_missed"] >= 2
).astype(int)

df["overdue_amount_proxy"] = (
    df["early_missed_installment_count"]
    * df["installment_amount"]
)

df["overdue_proportion"] = (
    df["overdue_amount_proxy"]
    / (
        df["prediction_installment"].clip(lower=1)
        * df["installment_amount"]
    )
).clip(0, 1)

candidate_numeric = [
    "early_missed_installment_count",
    "early_max_consecutive_missed",
    "early_max_overdue_days",
    "early_total_overdue_days",
    "early_recovery_delay_cycles",
    "pass_due_cycle_ratio",
    "overdue_installment_equivalent",
    "missed_installment_proportion",
    "overdue_proportion",
]

df[candidate_numeric + ["is_good_or_bad"]].head()

## 3. Point-biserial correlation for continuous behavioral signals

A point-biserial correlation evaluates the association between a continuous variable and a binary outcome.

Here it is used as a **screening and interpretation tool** to quantify the direction and strength of relationships between early behavioral measurements and eventual default.

A positive coefficient means higher values tend to be associated with the default class (`1`); a negative coefficient indicates the opposite direction.

In [ ]:
rows = []

for col in candidate_numeric:
    tmp = df[[col, "is_good_or_bad"]].dropna()

    if tmp[col].nunique() < 2:
        continue

    r, p = pointbiserialr(
        tmp["is_good_or_bad"].astype(int),
        tmp[col]
    )

    rows.append({
        "feature": col,
        "point_biserial_r": r,
        "p_value": p,
        "n": len(tmp),
        "abs_r": abs(r)
    })

point_biserial_results = (
    pd.DataFrame(rows)
    .sort_values("abs_r", ascending=False)
    .reset_index(drop=True)
)

point_biserial_results

### Interpretation

The ranking is useful for identifying candidate signals, but the coefficient magnitude should be interpreted in context.

A very small p-value can coexist with a modest effect size in a large dataset. Therefore, **statistical significance and practical importance are treated separately**.

In [ ]:
plt.figure(figsize=(9, 5))

plot_df = point_biserial_results.sort_values("point_biserial_r")

plt.barh(
    plot_df["feature"],
    plot_df["point_biserial_r"]
)

plt.axvline(0, linewidth=1)
plt.title("Point-Biserial Associations with Eventual Default")
plt.xlabel("Point-biserial correlation")
plt.ylabel("Candidate behavioral feature")
plt.tight_layout()
plt.show()

## 4. Interpreting the behavioral questions

The statistical analysis is deliberately connected back to the hypotheses from Notebook 02.

### H1 — Persistent early delinquency

Does consecutive missed-payment behavior have a stronger relationship with default than a simple miss count?

### H2 — Recovery persistence

Does a longer recovery delay correspond to higher eventual-default risk?

### H3 — Temporal normalization

Does overdue duration measured in repayment cycles retain an association with default?

### H4 — Relative delinquency burden

Does overdue amount relative to scheduled repayment associate with default?

### H5 — Normalized miss frequency

Does missed-installment proportion provide information beyond raw missed-installment count?

The tests above establish association. Predictive usefulness will be evaluated later.

## 5. Chi-square tests for categorical variables

Categorical dimensions can also contain systematic differences in default rates.

The original analytical work examined categorical variables such as repayment frequency, product, project, sector, and duplication indicators.

The public synthetic dataset contains a smaller set of analogous dimensions so the methodology can be reproduced without exposing production categories.

In [ ]:
# Build an abstracted categorical-analysis table.
categorical_features = ["frequency_name"]

# Add a few synthetic categorical variables so the procedure remains reproducible.
rng = np.random.default_rng(42)

df["product_group"] = rng.choice(
    ["Product_A", "Product_B", "Product_C", "Product_D"],
    size=len(df),
    p=[0.35, 0.30, 0.20, 0.15]
)

df["sector_group"] = rng.choice(
    ["Agriculture", "Trade", "Services", "Other"],
    size=len(df),
    p=[0.30, 0.30, 0.25, 0.15]
)

categorical_features += ["product_group", "sector_group"]

In [ ]:
chi_rows = []

for col in categorical_features:
    table = pd.crosstab(df[col], df["is_good_or_bad"])

    chi2, p, dof, expected = chi2_contingency(table)

    # Cramér's V as an effect-size companion to the p-value.
    n = table.to_numpy().sum()
    r, k = table.shape
    cramers_v = np.sqrt(
        (chi2 / n) /
        max(1, min(k - 1, r - 1))
    )

    chi_rows.append({
        "feature": col,
        "chi2": chi2,
        "degrees_of_freedom": dof,
        "p_value": p,
        "cramers_v": cramers_v
    })

chi_square_results = (
    pd.DataFrame(chi_rows)
    .sort_values("cramers_v", ascending=False)
    .reset_index(drop=True)
)

chi_square_results

### Why report an effect size?

With a large sample, even a weak association can become statistically significant.

Cramér's V is included here as a compact measure of association strength alongside the chi-square p-value.

This keeps the analysis focused on:

**Is there evidence of association?**

and

**How strong is the association?**

rather than using the p-value alone.

## 6. Contingency-table inspection

A significant chi-square statistic tells us that the distributions are not independent, but it does not tell us **where the difference occurs**.

Inspect category-level default rates to identify which groups contribute meaningful behavioral differences.

In [ ]:
def category_default_rates(data, feature):
    result = (
        data.groupby(feature, observed=True)["is_good_or_bad"]
             .agg(["count", "mean"])
             .rename(columns={"mean": "default_rate"})
             .reset_index()
    )
    result["default_rate_pct"] = (
        result["default_rate"] * 100
    ).round(2)
    return result.sort_values("default_rate_pct", ascending=False)

for feature in categorical_features:
    print(f"\n--- {feature} ---")
    display(category_default_rates(df, feature))

## 7. Adjusted Pearson residuals: locating the source of association

For a contingency table, adjusted Pearson residuals help identify cells where the observed count differs from the count expected under independence.

Conceptually:

- positive residual → more observations than expected;
- negative residual → fewer observations than expected.

This is useful when a categorical variable has several levels and the goal is not only to establish that an association exists, but also to understand **which categories drive it**.

In [ ]:
def adjusted_pearson_residuals(table):
    observed = table.to_numpy(dtype=float)

    row_totals = observed.sum(axis=1, keepdims=True)
    col_totals = observed.sum(axis=0, keepdims=True)
    total = observed.sum()

    expected = row_totals @ col_totals / total

    row_prop = row_totals / total
    col_prop = col_totals / total

    # Standard Pearson residual.
    pearson = (observed - expected) / np.sqrt(expected)

    # Adjusted residual using row/column marginal proportions.
    adjusted = pearson / np.sqrt(
        (1 - row_prop) @ (1 - col_prop)
    )

    return pd.DataFrame(
        adjusted,
        index=table.index,
        columns=table.columns
    )

for feature in categorical_features:
    table = pd.crosstab(df[feature], df["is_good_or_bad"])
    residuals = adjusted_pearson_residuals(table)

    print(f"\nAdjusted Pearson residuals — {feature}")
    display(residuals.round(2))

## 8. Research interpretation

This stage converts the exploratory patterns into statistically testable evidence.

The intended interpretation is:

### Behavioral variables

Point-biserial analysis identifies continuous early-life measures that have measurable associations with eventual default.

### Categorical variables

Chi-square analysis identifies dimensions whose outcome distributions differ systematically across categories.

### Residual analysis

Adjusted residuals help locate the categories responsible for those differences.

The results do **not** by themselves determine the final model. They inform:

- which representations deserve feature-engineering effort;
- which variables require further scrutiny;
- which categorical dimensions may contain useful heterogeneity;
- which hypotheses should be carried into predictive experiments.

## 9. Important methodological caution

This notebook intentionally does not perform automatic feature selection solely by p-value.

A feature can be statistically significant but:

- have a negligible practical effect;
- duplicate information contained in another variable;
- be unavailable at deployment time;
- introduce leakage;
- represent an unreliable operational field;
- become unstable across time or segments.

Therefore, statistical evidence is treated as **one source of evidence within the larger research process**.

## 10. Transition to feature engineering

The next stage will focus on the representation of behavioral variables.

Particular attention will be given to:

- raw versus normalized measures;
- proportions versus counts;
- temporal persistence;
- categorical encoding;
- high-cardinality geographic/product structure;
- feature redundancy and dimensionality.

The goal is to determine which statistically supported behavioral signals can be represented in a way that is both **predictively useful and operationally interpretable**.